In [3]:
"""
verify_mem_files.py  --  independent check of the exported design.

Reads ONLY config.json and the .mem files (not the Python model state), re-runs the integer pipeline from scratch and
compares every stage against the expected-output .mem files bit for bit:
    accumulators -> normalized features v -> scores -> decisions   (for every variant)
It also checks that every value fits its declared width and reports how the 4 stress shots behave.
This catches export bugs (wrong packing order, bad two's complement, wrong ordering) before any RTL is written.

Usage:  python verify_mem_files.py design_freeze          (verifies every mode folder inside)
"""
import os
import sys
import json
import numpy as np


def read_mem(path):
    vals = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("//"):
                vals.append(int(line, 16))
    return vals


def signed(x, bits):
    x &= (1 << bits) - 1
    return x - (1 << bits) if x >> (bits - 1) else x


def fits(arr, bits):
    return int(np.abs(arr).max()) < 2 ** (bits - 1) if arr.size else True


def verify(folder):
    cfg = json.load(open(os.path.join(folder, "config.json")))
    n_q, T = cfg["n_qubits"], cfg["T_bins"]
    wd = cfg["widths"]
    DLO, F, WB, MW = wd["dlo_bits"], wd["F_bits"], wd["w_bits"], wd["mem_word"]
    print(f"\n=== {cfg['mode']} === ({cfg['description']})")
    ok_all = True

    # ---- read tables and constants
    A = np.zeros((n_q, 4, T), np.int64)
    B = np.zeros((n_q, 4, T), np.int64)
    mu = np.zeros((n_q, 4), np.int64)
    n = np.zeros((n_q, 4), np.int64)
    for q in range(n_q):
        lines = read_mem(os.path.join(folder, f"dlo_q{q}.mem"))
        assert len(lines) == T, f"dlo_q{q}.mem has {len(lines)} lines, expected {T}"
        for t, v in enumerate(lines):
            for k in range(4):
                A[q, k, t] = signed(v >> ((2 * k) * DLO), DLO)
                B[q, k, t] = signed(v >> ((2 * k + 1) * DLO), DLO)
        mu[q] = [signed(x, MW) for x in read_mem(os.path.join(folder, f"norm_mu_q{q}.mem"))]
        n[q] = read_mem(os.path.join(folder, f"norm_n_q{q}.mem"))

    # ---- read testbench input and expected outputs
    gd = os.path.join(folder, "golden")
    words = read_mem(os.path.join(gd, "tb_input.mem"))
    G = len(words) // T
    I = np.array([signed(w & 0xFFFF, 16) for w in words], np.int64).reshape(G, T)
    Q = np.array([signed(w >> 16, 16) for w in words], np.int64).reshape(G, T)
    exp_acc = np.array([signed(x, MW) for x in read_mem(os.path.join(gd, "tb_acc.mem"))], np.int64).reshape(G, n_q, 4)
    exp_v = np.array([signed(x, 16) for x in read_mem(os.path.join(gd, "tb_v.mem"))], np.int64).reshape(G, n_q, 4)

    # ---- stage 1: accumulate
    acc = np.zeros((G, n_q, 4), np.int64)
    for q in range(n_q):
        for k in range(4):
            acc[:, q, k] = I @ A[q, k] + Q @ B[q, k]
    bad = int((acc != exp_acc).sum())
    print(f"accumulators      : {'PASS' if bad == 0 else f'FAIL ({bad} mismatches)'}  ({G} shots, incl. 4 stress)")
    ok_all &= bad == 0
    fit = fits(acc, wd["acc_bits_worst_case"])
    print(f"accumulator width : {'PASS' if fit else 'FAIL'} (fits {wd['acc_bits_worst_case']} signed bits)")
    ok_all &= fit

    # ---- stage 2: normalize
    one = np.int64(1)
    tmp = np.clip(acc + (one << n) - mu, 0, one << (n + 1))
    v = ((tmp << F) >> (n + 1)) - (one << (F - 1))
    bad = int((v != exp_v).sum())
    print(f"normalized v      : {'PASS' if bad == 0 else f'FAIL ({bad} mismatches)'}")
    ok_all &= bad == 0
    fit = fits(v, wd["feature_bits"] + 1) and np.abs(v).max() <= 2 ** (F - 1)
    print(f"feature range     : {'PASS' if fit else 'FAIL'} (|v| <= 2^{F - 1}; max seen {int(np.abs(v).max())})")
    ok_all &= bool(fit)
    st = v[-4:].reshape(4, -1)
    print(f"stress shots v    : max|v| per stress shot = {np.abs(st).max(axis=1).tolist()}  (saturation exercised if = {2 ** (F - 1)})")

    # ---- stages 3-4: per variant
    vf = v.reshape(G, -1)
    for variant, vc in cfg["variants"].items():
        exp_sc = np.array([signed(x, MW) for x in read_mem(os.path.join(gd, f"tb_score_{variant}.mem"))], np.int64).reshape(G, n_q)
        exp_pr = np.array(read_mem(os.path.join(gd, f"tb_pred_{variant}.mem")), np.int64).reshape(G, n_q)
        sc = np.zeros((G, n_q), np.int64)
        pr = np.zeros((G, n_q), np.int64)
        for q in range(n_q):
            w = np.array([signed(x, WB) for x in read_mem(os.path.join(folder, variant, f"cls_w_q{q}.mem"))], np.int64)
            thr = signed(read_mem(os.path.join(folder, variant, f"cls_thr_q{q}.mem"))[0], MW)
            pairs = vc["pairs"][q]
            cols = [np.full(G, 1 << F, np.int64)] + [vf[:, i] for i in range(vf.shape[1])]
            cols += [(vf[:, i] * vf[:, j]) >> F for i, j in pairs]
            O = np.stack(cols, 1)
            assert O.shape[1] == len(w), f"{variant} q{q}: {O.shape[1]} inputs vs {len(w)} weights"
            sc[:, q] = O @ w
            pr[:, q] = sc[:, q] > thr
        bad_s, bad_p = int((sc != exp_sc).sum()), int((pr != exp_pr).sum())
        fit = fits(sc, vc["score_bits"])
        print(f"[{variant:<4s}] scores     : {'PASS' if bad_s == 0 else f'FAIL ({bad_s})'}   decisions: "
              f"{'PASS' if bad_p == 0 else f'FAIL ({bad_p})'}   width {vc['score_bits']} bits: {'PASS' if fit else 'FAIL'}")
        ok_all &= bad_s == 0 and bad_p == 0 and bool(fit)
    return ok_all



if __name__ == "__main__":
    # Use the explicitly supplied folder only if it is valid.
    root = "design_freeze"

    if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
        root = sys.argv[1]

    if not os.path.isdir(root):
        raise SystemExit(f"Directory not found: {root}")

    folders = sorted(
        os.path.join(root, d)
        for d in os.listdir(root)
        if os.path.isfile(os.path.join(root, d, "config.json"))
    )

    if not folders:
        raise SystemExit(f"no config.json found under {root}")

    results = {f: verify(f) for f in folders}

    print("\n==================== VERIFY SUMMARY ====================")
    for f, ok in results.items():
        print(f"{f}: {'ALL BIT-EXACT' if ok else 'FAILURES - do not hand to RTL'}")

    



=== bus5 === (SYNTHETIC 5-qubit multiplexed bus built from real single-qubit shots, crosstalk strength 1.0)
accumulators      : PASS  (260 shots, incl. 4 stress)
accumulator width : PASS (fits 34 signed bits)
normalized v      : PASS
feature range     : PASS (|v| <= 2^11; max seen 2048)
stress shots v    : max|v| per stress shot = [2048, 2048, 287, 990]  (saturation exercised if = 2048)
[lin ] scores     : PASS   decisions: PASS   width 26 bits: PASS
[poly] scores     : PASS   decisions: PASS   width 27 bits: PASS

=== single === (REAL QICK single-qubit data (Zenodo 14427490), baseband (already downconverted))
accumulators      : PASS  (260 shots, incl. 4 stress)
accumulator width : PASS (fits 34 signed bits)
normalized v      : PASS
feature range     : PASS (|v| <= 2^11; max seen 2048)
stress shots v    : max|v| per stress shot = [2048, 2048, 294, 515]  (saturation exercised if = 2048)
[lin ] scores     : PASS   decisions: PASS   width 23 bits: PASS
[poly] scores     : PASS   decisio